# Steam Games EDA — artermiloff/steam-games-dataset (March 2025)

使用 ydata-profiling 對 `games_march2025_cleaned.csv` 進行完整 EDA。

In [ ]:
import pandas as pd
import numpy as np
import ast

df = pd.read_csv('/kaggle/input/steam-games-dataset/games_march2025_cleaned.csv', low_memory=False)
print('Shape:', df.shape)
df.head(3)

In [ ]:
# ---- 前處理：選 EDA 用的欄位，避免文字欄位干擾報告 ----
drop_cols = ['detailed_description', 'about_the_game', 'short_description',
             'reviews', 'header_image', 'website', 'support_url', 'support_email',
             'metacritic_url', 'notes', 'screenshots', 'movies', 'packages',
             'score_rank', 'full_audio_languages']

df_eda = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

# tags dict -> top tag (string)
def top_tag(val):
    try:
        d = ast.literal_eval(str(val))
        return max(d, key=d.get) if d else None
    except:
        return None

df_eda['top_tag'] = df_eda['tags'].apply(top_tag)
df_eda = df_eda.drop(columns=['tags'])

# genres / categories: 只取第一個
def first_item(val):
    try:
        lst = ast.literal_eval(str(val))
        return lst[0] if lst else None
    except:
        s = str(val).split(',')[0].strip().strip("'[]\"") if pd.notna(val) else None
        return s

df_eda['primary_genre'] = df_eda['genres'].apply(first_item)
df_eda['primary_category'] = df_eda['categories'].apply(first_item)
df_eda = df_eda.drop(columns=['genres', 'categories', 'developers', 'publishers',
                               'supported_languages', 'user_score'])

# estimated_owners: 取中間值
def owners_mid(val):
    try:
        parts = str(val).replace(',','').split(' - ')
        return (int(parts[0]) + int(parts[1])) / 2
    except:
        return None

df_eda['estimated_owners_mid'] = df_eda['estimated_owners'].apply(owners_mid)
df_eda = df_eda.drop(columns=['estimated_owners'])

# release_date -> year
df_eda['release_year'] = pd.to_datetime(df_eda['release_date'], errors='coerce').dt.year
df_eda = df_eda.drop(columns=['release_date'])

print('EDA dataframe shape:', df_eda.shape)
df_eda.dtypes

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df_eda,
    title='Steam Games EDA (artermiloff March 2025)',
    explorative=True,
    minimal=False
)

profile.to_file('steam_eda_report.html')
print('Report saved to steam_eda_report.html')

In [ ]:
profile.to_notebook_iframe()